# 03 - Camera Intrinsics and Extrinsics

## Imports

In [1]:
import numpy as np
import pandas as pd

## Why Are We Talking about Cameras (Again)?

In our Computer Vision primer, we covered some of the basics of a camera and how it leverages light to create an image. This understanding covered parts of the basics of camera **intrinsics**, or the internals of the camera that tells us how it sees the world. 

For us to understand how to use Limelights (or any other cameras), particularly more than one, to perform object detection tasks, we need to not only have a strong understanding of the intrinsics, but also the camera **extrinsics**, or where the camera is located in the real world. We need to know the location of all of our cameras so we can use them together! 

It is important to note that we will be discussing a bit more math-intensive topics - do not stress if the math is a little heavier! The main point is to give you the foundational understanding, so if you get the big ideas, you are in good shape. 

#### Perspective Projection Recap

Recall from our discussion about perspective projection that the fundamental operation a camera does is mapping points in the real world (which is 3D) to points on a piece of paper (which is 2D), also known as perspective projection. 

From the pinhole camera geometry below:


![](../cv_primer/ref_imgs/projection_diagram.png)

We were able to recreate the perspective projection equations:

$x = f\frac{X}{Z}$, $y = f\frac{Y}{Z}$

Revisit the notebook `05-perspective-projection.ipynb` if you want to recap this information more deeply - it is worth the extra time.

What we will be walking through in this notebook is how we can relate our perspective projection to both the intrinsics and extrinsics of a camera. That is, how understanding both what's inside and outside of a camera will help us understand the position of objects in images. 

## Intrinsics

### Intrinsic Matrix

As shown by our above diagram, we map 3D points from the perspective of the camera to 2D points on an image. Our intrinsic matrix is what helps us perform this mapping, using a matrix 

$$
K = 
\begin{bmatrix}
f_x & s & c_x \\
0 & f_y & c_y \\
0 & 0 & 1
\end{bmatrix}
$$ 

Where $(c_x, c_y)$ is our **principal point** ($p$), $(f_x, f_y)$ are our **focal lengths**, where we have two different ones for $x$ and $y$ to account for some potential non-square pixels, and our **skew** $s$ which corrects for non-perpendicular axes between the camera sensor and the camera lens. We typically assume $s = 0$ (no non-perpendicular axes) and $f_x = f_y$ (no non-square pixels) for simplicity by reducing the number of variables we deal with in our matrix multiplication, leaving us with 

$$
K = 
\begin{bmatrix}
f & 0 & c_x \\
0 & f & c_y \\
0 & 0 & 1
\end{bmatrix}
$$ 

Note that typically our principal point is also $(0, 0)$ in a lot of simplified applications, but for our use cases, this simply isn't a realistic assumption given the amount of movement and different cameras we use. 

Let's look at an example matrix below: 


In [2]:
sample_k = np.array([[800, 0, 320], [0, 800, 240], [0, 0, 1]])
sample_k

array([[800,   0, 320],
       [  0, 800, 240],
       [  0,   0,   1]])

A zoom of around 800 means our image is moderately zoomed in, and if our image resolution is $640 \times 480$, then that places our principal point roughly in the center of our image (which is not terribly uncommon).

### Mapping from 3D Camera World to 2D Image 

We can actually use this intrinsic matrix to map 3D points in our camera coordinates to the 2D image. We use the following equation: 

$p = KP$

where 

$p = 
\begin{bmatrix}
u \\
v \\ 
1
\end{bmatrix}
$,
$P =
\begin{bmatrix}
X_c \\
Y_c \\
Z_c \\
\end{bmatrix}
$, 
$K = 
\begin{bmatrix}
f_x & s & c_x \\
0 & f_y & c_y \\
0 & 0 & 1
\end{bmatrix}
$

with $p$ as our point in the 2D plane, $K$ our intrinsic matrix, and $P$ as our point in the 3D plane (with the $c$ subscript reminding us that we are dealing with coordinates with respect to the camera). Note that the $1$ in $p$ is for making the matrix multiplication work properly because our matrix is not square (don't worry about why for building intuition, but feel free to look this up on your free time!).

We can write out this equation out as 

$$
\begin{bmatrix}
u \\
v \\ 
1
\end{bmatrix} = 
\begin{bmatrix}
f_x & s & c_x \\
0 & f_y & c_y \\
0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
X_c \\
Y_c \\
Z_c \\
\end{bmatrix}
$$ 

If we actually do this multiplication, we get the following representations of our 2D coordinates as 

$$
u = f_x \frac{X_c}{Z_c} + c_x
$$, 
$$
v = f_y \frac{Y_c}{Z_c} + c_y
$$

These are our perspecitve projection equations, taking into account the principal point! 

So, we have shown a bit more formally how the camera's intrinsic properties determine perspective projection. 





## Extrinsics

### Extrinsic Matrix

The intrinsic matrix above uses "camera coordinates", which we denoted with the $c$ subscript. Obviously, it is important for us to understand real-world objects from the perspective of the camera. However, we also need to understand where those objects are in the real-world, regardless of the perspective of the camera! 

We turn to an extrinsic matrix to help us map "world coordinates" (subscript $w$) to "camera coordinates", defined as 

$$ M = [R | t] $$

where 

$$ R =
\begin{bmatrix}
r_{11} & r_{12} & r_{13} \\
r_{21} & r_{22} & r_{23} \\
r_{31} & r_{32} & r_{33} \\
\end{bmatrix}
,
t = 
\begin{bmatrix}
t_x \\
t_y \\
t_z \\
\end{bmatrix}
$$

and therefore we have 

$$ M = 

\left[
\begin{array}{ccc|c}
r_{11} & r_{12} & r_{13} & t_x \\
r_{21} & r_{22} & r_{23} & t_y \\
r_{31} & r_{32} & r_{33} & t_z \\
\end{array}
\right]
$$

The notation $[R | t]$ represents an augmented matrix, which is the matrix form of representing linear equations. So, something like $Mx$ would be written as $Rx + t$.


What do $R$ and $t$ tell us? 

#### Rotation Matrix

A **rotation matrix** rotates points in space by a specific angle without changing the size or shape of those points. A rotation can look like this: 

![](./ref_imgs/3d_rotation.png)

Naturally, we can do a number of different rotations around the different axes. 

As for what the actual values represent, they simply represent matrix multiplications that allow that rotation to take place. Differing values can create different rotations. For instance, a matrix like this one:

$$
R = 
\begin{bmatrix}
0 & 0 & -1 \\
0 & 1 & 0 \\
1 & 0 & 0 \\
\end{bmatrix}
$$

would produce the rotation we saw in the above figure. 

The top row dictates the direction of the new x-axis, the middle the y-axis, and the bottom the z-axis. 

There are a *ton* of different rotation matrices, and there is a good amount of matrix operations you would have to learn to do them properly (and understanding basis vectors!). Feel free to take time to undestand the math if you so choose, but the biggest thing here is to understand underlying intuition.  


#### Translation Matrix

A **translation matrix** moves points in space by the same amount in a given direction, without changing the size or shape of those points. Note that while we call this a matrix, because it is 1 dimensional, it can also be referred to as a vector.

 A translation can look like this: 

![](./ref_imgs/3d_translation.png)


Just like rotations, we can do a number of different translations. 

The values are a little simpler here, since our $t$ matrix is $3 \times 1$ and our $R$ is $3 \times 3$. The top value represents the amount to move a point along the x-axis, the middle along the y-axis, and the bottom along the z-axis. 

For example, a matrix like this one 

$$
t = 
\begin{bmatrix}
0 \\
1 \\
0 \\
\end{bmatrix}
$$

will produce the above transformation. 

### Mapping 3D Real World to 3D Camera 

So, now that we understand the components of $M$, how does this help us go from the real-world coordinates of an object to the camera coordinates? 

$R$ tells us the rotation of our camera, and $T$ tells us the translation of the camera, both of which are relative to the real world position of the field of view that contains our object. 

We use the equation 

$$ P_c = M P_w$$

where

$$
P_c = 
\begin{bmatrix}
X_c \\
Y_c \\
Z_c \\
\end{bmatrix}, 
P_w = 
\begin{bmatrix}
X_w \\
Y_w \\
Z_w \\
1
\end{bmatrix}, 
$$

and $M$ is our same extrinsic matrix, leaving us with 

$$
\begin{bmatrix}
X_c \\
Y_c \\
Z_c \\
\end{bmatrix} = 
\left[
\begin{array}{ccc|c}
r_{11} & r_{12} & r_{13} & t_x \\
r_{21} & r_{22} & r_{23} & t_y \\
r_{31} & r_{32} & r_{33} & t_z \\
\end{array}
\right]
\begin{bmatrix}
X_w \\
Y_w \\
Z_w \\
1
\end{bmatrix}
$$





### Perspective Projection in 3D 

We have now seen that with our intrinsic matrix, we can take our coordinates from the 3D camera to the 2D image. With our extrinsitc matrix, we can take our coordinates from the 3D world to the 3D camera. Unsurprisingly, we can combine the two to give us the full mapping of 3D points in the world to the 2D image.

The equation is 
$$p = \lambda KMP$$

where $\lambda$ represents the depth of our object in camera coordinates and $P$ is $P_w$ from our extrinsic equation. 

The only different term is $p$, where 

$$
p = 
\begin{bmatrix}
x_s \\
y_s \\
s
\end{bmatrix}
$$

such that $x = \frac{x_s}{s}$ and $y = \frac{y_s}{s}$, representing our 2D coordinates! 

Now you have seen how the inside and outside of the camera determine how a 3D object gets mapped to a 2D image. 


But, we have a slight problem - how do we figure out this value of $\lambda$?